# Which channel is losing Kalpa's consumers, once the Business orders are read apart?

**Week 2, Monday. The second case, in pairs or alone.** It runs in the take-home.

> "The same numbers for every channel: app, web and store. Marketing says the store is booming
> and the web is collapsing, and wants the budget moved. Is that what the book says?"
> Anand Iyer, finance controller, Kalpa Retail

**The situation.** Kalpa sells through three channels: its app, its website and its stores.
Booked revenue, every order at its amount, was Rs 10,00,00,000 in Q1 (April to June 2026) and
Rs 9,84,00,000 in Q2 (July to September 2026). Business, the corporate segment, books about 99
percent of those rupees in orders worth lakhs each, while the other three segments, Retail-Core,
Retail-Plus and Student, are Kalpa's consumers, whose orders are worth hundreds or a few thousand.
Marketing read the channel totals and wants budget moved from the web to the stores.

**How to answer.** Each step holds `TODO` markers: lettered choices in a comment above a
placeholder whose name starts with `__TODO` and ends in the marker's number. Replace each
placeholder with the letter in quotes and run the step; the check after it tells you whether the
step behaves. Running the notebook before a marker is filled stops at that marker with a
`NameError`, as expected. Post your seven letters in order.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

import re
from decimal import Decimal
SQL_DIR = kit.data_dir().parent / "sql"

def blocks(stem):
    """Every named block of a .sql file in ../sql/, keyed by the name on its '-- name:' line."""
    text = (SQL_DIR / f"C2_W02_D01_{stem}_STUDENT.sql").read_text(encoding="utf-8")
    parts = re.split(r"^-- name: (\w+)\s*$", text, flags=re.M)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

def statements(block):
    """A block's statements with its comment lines removed, for a block that runs several."""
    bare = "\n".join(l for l in block.splitlines() if not l.strip().startswith("--"))
    return [s.strip() for s in bare.split(";") if s.strip()]

def num(v):
    """A database number as a Python int or float."""
    if isinstance(v, Decimal):
        return int(v) if v == v.to_integral_value() else float(v)
    return v

def show(rows, caption="", money=(), limit=12):
    """Rows from kit.sql as the kit's table, with the columns named in money set in rupees."""
    if not rows:
        kit.table(["result"], [["no rows"]], caption)
        return
    heads = list(rows[0])
    def cell(h, v):
        v = num(v)
        if v is None:
            return "NULL"
        if h in money:
            return kit.rupees(v)
        return f"{v:,}" if isinstance(v, int) and abs(v) >= 10000 else str(v)
    body = [[cell(h, r[h]) for h in heads] for r in rows[:limit]]
    if len(rows) > limit:
        body.append(["..." for _ in heads])
    kit.table(heads, body, caption)

def run(name, caption="", money=(), limit=12):
    """Print a named block, run it, show its rows and hand them back."""
    print(Q[name])
    rows = kit.sql(Q[name])
    show(rows, caption, money, limit)
    return rows

def pct(before, after):
    """The change from before to after, in percent."""
    return 100 * (float(after) - float(before)) / float(before)

def rows(query):
    """Run a query and hand back its rows with every number as a Python number."""
    return [{k: num(v) for k, v in r.items()} for r in kit.sql(query)]
print("connected:", kit.sql("select version()")[0]["version"].split(",")[0])

In [ ]:
kit.side_by_side(
    kit.ladder(["The channel totals", "Business and consumers apart", "The consumers' tree per channel",
                "The tie-out", "The line for Anand"], lit=0, show=False),
    kit.driver_tree({"label": "a channel's revenue", "kind": "lit", "children": [
        {"label": "Business", "note": "a few large orders"},
        {"label": "consumers", "note": "many small orders"}]}, show=False),
)

## Step 1. What do the channel totals say from Q1 to Q2?

Where this is used at work: a channel manager's budget follows the channel's total, so the
total is the first number anyone reads.

In [ ]:
# TODO 1. Which grouping gives one row per channel and quarter?
#   a) GROUP BY channel, quarter
#   b) GROUP BY channel
#   c) GROUP BY quarter
#   d) GROUP BY customer_id, channel
GROUPING = {"a": "GROUP BY channel, quarter", "b": "GROUP BY channel", "c": "GROUP BY quarter",
            "d": "GROUP BY customer_id, channel"}[__TODO1__]
totals = rows(f"SELECT channel, quarter, count(*) AS orders, sum(amount) AS revenue FROM orders {GROUPING} ORDER BY 1, 2")
show(totals, "The channel totals", money=("revenue",))
t = {(r["channel"], r["quarter"]): r for r in totals}
channels = sorted({r["channel"] for r in totals})
kit.columns(channels, [(q, [t[(c, q)]["revenue"] / 1e5 for c in channels]) for q in ("Q1", "Q2")],
            title="Booked revenue per channel, Rs lakh", fmt=lambda v: f"{v:,.0f}")

In [ ]:
kit.check("six rows, one per channel and quarter", len(totals) == 6)
kit.check("the channels add back to the book in each quarter",
          sum(t[(c, "Q1")]["revenue"] for c in channels) == 100000000
          and sum(t[(c, "Q2")]["revenue"] for c in channels) == 98400000)

## Step 2. How much of each channel is Business, and how much is consumers?

Where this is used at work: a total made of a few very large orders and many small ones is two
businesses, and each is read on its own before the total is.

In [ ]:
# TODO 2. Which expression labels each order as Business or consumer?
#   a) c.segment
#   b) CASE WHEN c.segment = 'Business' THEN 'Business' ELSE 'consumer' END
#   c) CASE WHEN o.amount > 100000 THEN 'Business' ELSE 'consumer' END
#   d) WHERE c.segment <> 'Business'
KIND = {"a": "c.segment", "b": "CASE WHEN c.segment = 'Business' THEN 'Business' ELSE 'consumer' END",
        "c": "CASE WHEN o.amount > 100000 THEN 'Business' ELSE 'consumer' END",
        "d": "WHERE c.segment <> 'Business'"}[__TODO2__]
split = rows(f"""SELECT o.channel, {KIND} AS kind, o.quarter, count(*) AS orders,
                        count(DISTINCT o.customer_id) AS customers, sum(o.amount) AS revenue
                 FROM orders o JOIN customers c USING (customer_id)
                 GROUP BY 1, 2, 3 ORDER BY 1, 2, 3""")
show(split, "Each channel, Business and consumers apart", money=("revenue",))

In [ ]:
kinds = {r["kind"] for r in split}
biz = {(r["channel"], r["quarter"]): r["revenue"] for r in split if r["kind"] == "Business"}
if biz:
    kit.columns(channels, [(q, [biz.get((c, q), 0) / 1e5 for c in channels]) for q in ("Q1", "Q2")],
                title="Business revenue per channel, Rs lakh: where the large orders went", fmt=lambda v: f"{v:,.0f}")
kit.check("every order is labelled one of two kinds", len(kinds) == 2, ", ".join(sorted(kinds)))
kit.check("the two kinds add back to each channel's total",
          all(sum(r["revenue"] for r in split if r["channel"] == c and r["quarter"] == q) == t[(c, q)]["revenue"]
              for c in channels for q in ("Q1", "Q2")))

## Step 3. How did each channel's consumers move, branch by branch?

Where this is used at work: a channel's consumer health is its consumers' tree, customers times
orders each times revenue per order, read as a change.

In [ ]:
# TODO 3. Which expression counts each channel's consumers, each once?
#   a) count(*)
#   b) sum(1)
#   c) count(DISTINCT o.customer_id)
#   d) count(o.customer_id)
CUST = {"a": "count(*)", "b": "sum(1)", "c": "count(DISTINCT o.customer_id)",
        "d": "count(o.customer_id)"}[__TODO3__]

# TODO 4. Which expression gives orders per consumer that multiplies back?
#   a) count(*) / {cust}
#   b) round(count(*) / {cust}, 2)
#   c) round({cust}::numeric / count(*), 2)
#   d) round(count(*)::numeric / {cust}, 2)
FREQ = {"a": "count(*) / {cust}", "b": "round(count(*) / {cust}, 2)", "c": "round({cust}::numeric / count(*), 2)",
        "d": "round(count(*)::numeric / {cust}, 2)"}[__TODO4__].format(cust=CUST)
consumers = rows(f"""SELECT o.channel, o.quarter, count(*) AS orders, {CUST} AS customers,
                            {FREQ} AS orders_per_customer, sum(o.amount) AS revenue
                     FROM orders o JOIN customers c USING (customer_id)
                     WHERE c.segment <> 'Business'
                     GROUP BY o.channel, o.quarter ORDER BY 1, 2""")
show(consumers, "Consumers only, per channel and quarter", money=("revenue",))
cons = {(r["channel"], r["quarter"]): r for r in consumers}
kit.columns(channels, [(q, [cons[(c, q)]["revenue"] / 1000 for c in channels]) for q in ("Q1", "Q2")],
            title="Consumer revenue per channel, Rs thousand", fmt=lambda v: f"{v:,.0f}")

In [ ]:
kit.check("consumers are fewer than their orders in every row", all(r["customers"] < r["orders"] for r in consumers))
kit.check("every ratio multiplies back to its orders within half an order",
          all(abs(float(r["orders_per_customer"]) * r["customers"] - r["orders"]) < 0.5 for r in consumers))

## Step 4. Which line goes on Anand's channel sheet?

Where this is used at work: the line a budget is moved on has to read the part of the business
the budget is for.

In [ ]:
change = {c: pct(cons[(c, "Q1")]["revenue"], cons[(c, "Q2")]["revenue"]) for c in channels}
total_change = {c: pct(t[(c, "Q1")]["revenue"], t[(c, "Q2")]["revenue"]) for c in channels}
kit.table(["channel", "booked total, Q1 to Q2", "consumers only, Q1 to Q2"],
          [(c, f"{total_change[c]:+.1f}%", f"{change[c]:+.1f}%") for c in channels],
          caption="The channel totals beside the consumers' own change")
kit.bars([(c, abs(change[c])) for c in channels], fmt=lambda v: f"down {v:.1f}%",
         title="How far each channel's consumer revenue fell, Q1 to Q2")

# TODO 5. Which channel lost the largest share of its consumer revenue?
#   a) web
#   b) store
#   c) none: every channel held its consumers
#   d) app
WORST = {"a": "web", "b": "store", "c": None, "d": "app"}[__TODO5__]

# TODO 6. Which line goes on Anand's sheet for the store?
#   a) Store revenue rose 61.1 percent: move budget to the stores.
#   b) Store consumer revenue fell 18.8 percent; the store total rose on Business orders.
#   c) The store is flat once the Business orders are removed.
#   d) Store revenue cannot be reported until Business is removed from the book.
LINE = {"a": 61.1, "b": -18.8, "c": 0.0, "d": None}[__TODO6__]

# TODO 7. Which fact would move the budget question back to the channel totals?
#   a) If the web's consumer orders fell further next quarter
#   b) If Anand asked for the channels in rupees rather than orders
#   c) If Business placed its orders through whichever channel its buyer chose on the day
#   d) If Business orders were placed through a channel because of that channel's own service
REASON = {"a": "consumers", "b": "rupees", "c": "chance", "d": "service"}[__TODO7__]

In [ ]:
kit.check("no channel lost a larger share of its consumer revenue than the one named",
          WORST is not None and all(change[WORST] <= v for v in change.values()))
kit.check("the store line carries the store's consumer change", LINE is not None and abs(LINE - change["store"]) < 0.05)
kit.check("every channel's consumers fell", all(v < 0 for v in change.values()))

**Post your answer.** Seven letters, in the order of the markers, and one line for Anand that
names the channel losing its consumers fastest, with its number.

In [ ]:
kit.check_summary()